# Lesson 2.3 — Prompt Techniques: Zero-shot, Few-shot, Chain-of-Thought

**Level: Intermediate**

## Learning Objectives

By the end of this lesson, you will be able to:

- Apply zero-shot, few-shot, and chain-of-thought prompting appropriately
- Choose a technique based on task type and cost/latency budget
- Build a small harness that compares techniques on the same task

**You will produce:** A comparison harness printing outputs of three prompting techniques side by side.

## Concepts

Three core techniques:

- **Zero-shot** — just instructions, no examples. Cheapest; works when the task is unambiguous.
- **Few-shot** — include a few input/output **examples** to pin format and edge-case handling. Costs more tokens but sharply improves consistency on structured tasks.
- **Chain-of-thought (CoT)** — ask the model to reason step by step before answering. Improves accuracy on multi-step/logical tasks at the cost of more output tokens and latency.

The architect's judgment: use the **least expensive technique that hits the quality bar**. Few-shot for formatting consistency; CoT for genuine reasoning; keep hidden reasoning out of the final payload when you only need the answer. These choices directly affect the token budget we optimize in Lesson 2.4.

> **Reference:** [Anthropic — Prompt engineering techniques](https://docs.anthropic.com/en/docs/build-with-claude/prompt-engineering/overview)

In [ ]:
# !pip install anthropic
import os

try:
    import anthropic
    if os.environ.get("ANTHROPIC_API_KEY"):
        client = anthropic.Anthropic(); LIVE = True
    else:
        client = None; LIVE = False
except Exception as e:
    client = None; LIVE = False
    print("anthropic SDK unavailable:", e)

# Confirm exact model IDs at https://docs.anthropic.com/en/docs/about-claude/models
MODELS = {
    "opus": "claude-opus-4-20250514",
    "sonnet": "claude-sonnet-4-20250514",
    "haiku": "claude-3-5-haiku-20241022",
}

def ask_claude(prompt, model=MODELS["sonnet"], system=None, max_tokens=512):
    if not LIVE:
        return f"[MOCK {model}] would answer: {prompt[:80]}..."
    msg = client.messages.create(model=model, max_tokens=max_tokens,
        system=system or "You are a helpful assistant.",
        messages=[{"role": "user", "content": prompt}])
    return msg.content[0].text

print("LIVE" if LIVE else "MOCK", "mode ready")

## Code-Along: Step 1 — Build the three prompt variants

We express each technique as a function returning the user prompt for the same classification task (triage urgency), so the only thing that varies is the technique.

In [ ]:
TASK = "Classify this support message urgency as low/medium/high: 'Our checkout is down for all users.'"

def zero_shot():
    return TASK

def few_shot():
    examples = (
        "Examples:\n"
        "msg: 'typo on the about page' -> low\n"
        "msg: 'login slow for some users' -> medium\n"
        "msg: 'payments failing for everyone' -> high\n\n"
    )
    return examples + TASK + " Answer with one word."

def chain_of_thought():
    return TASK + " First reason briefly about blast radius and revenue impact, then give the label."

print(few_shot())

## Code-Along: Step 2 — Run the comparison harness

We call all three and print results together. In mock mode you still see how the prompts differ; with a key you see real quality differences.

In [ ]:
techniques = {"zero_shot": zero_shot(), "few_shot": few_shot(), "cot": chain_of_thought()}
for name, prompt in techniques.items():
    out = ask_claude(prompt, model=MODELS["haiku"], max_tokens=200)
    print(f"=== {name} ===\n{out}\n")

## Exercise

Pick a *reasoning-heavy* task (e.g. 'does this refund request violate the 30-day policy given this timeline?'). Show that zero-shot is unreliable while CoT is correct, then write one sentence on the token/latency cost you accepted for the accuracy gain.

In [ ]:
# TODO: define a reasoning-heavy TASK2
# TODO: build zero-shot and CoT prompts and run them through ask_claude
# TODO: comment on the accuracy vs token/latency trade-off you observed

## Key Takeaways

- Zero-shot is cheapest; few-shot pins format; CoT boosts multi-step reasoning.
- Pick the least expensive technique that clears the quality bar.
- Few-shot examples are powerful but add input tokens on every call.
- CoT trades output tokens and latency for accuracy — budget for it deliberately.

---

**Next:** [Lesson 2.4 — Context Windows, Token Budgets, and Prompt Reuse](./lesson-2.4-context-tokens-caching.ipynb)